# 🛰️ SpaceLAB · Hackatón FAC+PUJ · GeoVision Colombia
## Baseline 2: detección de embarcaciones con Faster R-CNN (Colab / Kaggle)

Un detector clásico de dos etapas, en **PyTorch puro** (torchvision, sin Ultralytics), con pesos preentrenados en COCO.
Hace el mismo ciclo que el baseline de YOLO:
1. Entrena un Faster R-CNN sobre `images/train` (lee directamente las etiquetas en formato YOLO).
2. Calcula el **puntaje oficial local** en `images/test`: `0.7·mAP@0.5:0.95 + 0.3·F1macro@0.5`.
3. Genera el **ZIP de envío** sobre `images/oculto`, listo para subir al leaderboard.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*. En Kaggle: *Settings → Accelerator → GPU T4*, y agregue el dataset con *Add Input*.

> Los baselines son solo puntos de partida para verificar que su flujo funciona de punta a punta. No son una recomendación: pueden usar cualquier modelo o framework (MMDetection, Detectron2, torchvision, RT-DETR, cualquier versión de YOLO, etc.). `formato_envio.py` les ayuda a generar el ZIP desde cualquiera de ellos.

In [ ]:
!nvidia-smi -L || echo 'Sin GPU: active la GPU en la configuración del entorno'
import torch, torchvision
print('torch', torch.__version__, '· torchvision', torchvision.__version__, '· GPU:', torch.cuda.is_available())

### 1. Datos
* **Colab:** suba `spacelab_dataset.zip` a su Google Drive (o agregue un acceso directo a la carpeta compartida por la organización) y ajuste `RUTA_ZIP_DRIVE`.
* **Kaggle:** agregue el dataset como *Input*; el cuaderno lo busca solo en `/kaggle/input`.
* **Local:** ponga en `DATOS` la ruta a la carpeta `spacelab_dataset`.

In [ ]:
import os, glob, subprocess
RUTA_ZIP_DRIVE = '/content/drive/MyDrive/spacelab_dataset.zip'   # <- ajuste si es necesario
DATOS = None

if os.path.exists('/kaggle/input'):                                # Kaggle
    yamls = glob.glob('/kaggle/input/**/spacelab_dataset/data.yaml', recursive=True)
    DATOS = os.path.dirname(yamls[0]) if yamls else None
elif os.path.exists('/content'):                                   # Colab
    if not os.path.exists('/content/spacelab_dataset/data.yaml'):
        from google.colab import drive
        drive.mount('/content/drive')
        print('Descomprimiendo (2-4 min)...')
        subprocess.run(['unzip', '-q', '-o', RUTA_ZIP_DRIVE, '-d', '/content/'], check=True)
    DATOS = '/content/spacelab_dataset'
else:                                                              # local
    DATOS = './spacelab_dataset'

assert DATOS and os.path.exists(os.path.join(DATOS, 'data.yaml')), 'No se encontró spacelab_dataset/data.yaml'
for s in ['train', 'test', 'oculto']:
    print(s, len(os.listdir(os.path.join(DATOS, 'images', s))), 'imágenes')
DATOS

### 2. Scripts del kit (evaluador oficial + ayuda de formato + baseline)

In [ ]:
%%writefile evaluar.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
evaluar.py — Evaluador oficial · SpaceLAB Hackatón FAC+PUJ · GeoVision Colombia
================================================================================

Puntaje oficial
---------------
    Score = 0.7 · mAP@0.5:0.95  +  0.3 · F1macro@0.5

* mAP@0.5:0.95  Promedio (macro) sobre las 5 clases del AP estilo COCO, calculado en
                10 umbrales de IoU (0.50, 0.55, ..., 0.95) con interpolación de 101 puntos.
                Usa TODAS las cajas enviadas (máximo 300 por imagen, las de mayor confianza).
* F1macro@0.5   Promedio (macro) sobre las 5 clases del F1 con IoU ≥ 0.5, usando SOLO las
                cajas con confianza ≥ umbral del equipo (archivo `umbral.txt` dentro del envío;
                por defecto 0.25). Mide qué tan "desplegable" es el sistema: cuántos barcos
                encuentra y cuántas falsas alarmas produce.
* Desempate     mAP@0.5 y, si persiste, el envío más temprano.

Formato del envío
-----------------
Un ZIP (o una carpeta) con un archivo .txt por imagen, con el MISMO nombre de la imagen
(p. ej. `sl_3fa9c2d1.jpg` -> `sl_3fa9c2d1.txt`). Una línea por detección, formato YOLO:

    clase x_centro y_centro ancho alto confianza

con coordenadas normalizadas entre 0 y 1 (relativas al ancho/alto de la imagen) y clase
entera 0..4 (0 small_boat, 1 cargo_ship, 2 fishing_vessel, 3 warship, 4 work_vessel).
Es exactamente lo que produce Ultralytics con `save_txt=True, save_conf=True`.
Imágenes sin archivo = sin detecciones. Opcional: `umbral.txt` con un número (p. ej. 0.30).

Uso
---
  # Participantes: evaluar sus predicciones sobre el split `test` (que tiene etiquetas)
  python evaluar.py --gt spacelab_dataset/labels/test --envio mis_predicciones.zip

  # Organizadores: un envío contra el conjunto oculto (público / privado)
  python evaluar.py --gt privado/gt_oculto/labels --meta privado/gt_oculto/meta.csv --envio equipo.zip

  # Organizadores: ranking de todos los envíos de una carpeta (un ZIP por equipo)
  python evaluar.py --gt privado/gt_oculto/labels --meta privado/gt_oculto/meta.csv \
                    --envios carpeta_envios/ --ranking ranking.csv

Solo depende de numpy.
"""
from __future__ import annotations

import argparse
import csv
import io
import json
import os
import sys
import zipfile
from dataclasses import dataclass, field

import numpy as np

CLASES = ["small_boat", "cargo_ship", "fishing_vessel", "warship", "work_vessel"]
NC = len(CLASES)
UMBRALES_IOU = np.linspace(0.5, 0.95, 10)
PUNTOS_RECALL = np.linspace(0.0, 1.0, 101)
MAX_DET = 300
UMBRAL_DEFECTO = 0.25
PESO_MAP, PESO_F1 = 0.7, 0.3


# --------------------------------------------------------------------------- lectura
@dataclass
class Envio:
    preds: dict            # id_imagen -> np.ndarray (N, 6): cls, xc, yc, w, h, conf
    umbral: float = UMBRAL_DEFECTO
    umbral_declarado: bool = False
    avisos: list = field(default_factory=list)


def _parse_lineas(texto: str, nombre: str, con_conf: bool, avisos: list) -> np.ndarray:
    filas = []
    for i, linea in enumerate(texto.splitlines(), 1):
        p = linea.replace(",", " ").split()
        if not p:
            continue
        if len(p) not in (5, 6):
            avisos.append(f"{nombre}:{i}: se esperaban 6 valores, hay {len(p)} -> línea ignorada")
            continue
        try:
            c = int(float(p[0]))
            xc, yc, w, h = (float(v) for v in p[1:5])
            conf = float(p[5]) if len(p) == 6 else 1.0
        except ValueError:
            avisos.append(f"{nombre}:{i}: valores no numéricos -> línea ignorada")
            continue
        if len(p) == 5 and con_conf:
            avisos.append(f"{nombre}: líneas sin confianza; se asume 1.0 (perjudica el mAP)")
            con_conf = False  # avisar solo una vez por archivo
        if not 0 <= c < NC:
            avisos.append(f"{nombre}:{i}: clase {c} fuera de rango 0..{NC-1} -> línea ignorada")
            continue
        if not all(np.isfinite([xc, yc, w, h, conf])):
            avisos.append(f"{nombre}:{i}: valores no finitos -> línea ignorada")
            continue
        if w <= 0 or h <= 0:  # cajas degeneradas: se ignoran y se reportan en un solo aviso
            avisos.append("__degenerada__")
            continue
        filas.append((c, xc, yc, w, h, conf))
    return np.asarray(filas, dtype=np.float64).reshape(-1, 6)


def _es_basura(ruta: str) -> bool:
    base = os.path.basename(ruta)
    return ("__MACOSX" in ruta or base.startswith("._") or base.startswith(".")
            or not base.lower().endswith(".txt"))


def leer_envio(ruta: str) -> Envio:
    """Lee un envío desde un .zip o una carpeta. Busca .txt en cualquier subcarpeta."""
    archivos = {}  # nombre_base -> texto
    if os.path.isdir(ruta):
        for raiz, _, fs in os.walk(ruta):
            for f in fs:
                r = os.path.join(raiz, f)
                if not _es_basura(r):
                    with open(r, "r", encoding="utf-8", errors="replace") as fh:
                        archivos.setdefault(f, fh.read())
    elif zipfile.is_zipfile(ruta):
        with zipfile.ZipFile(ruta) as z:
            for info in z.infolist():
                if info.is_dir() or _es_basura(info.filename):
                    continue
                if info.file_size > 50_000_000:
                    raise ValueError(f"Archivo demasiado grande dentro del ZIP: {info.filename}")
                archivos.setdefault(os.path.basename(info.filename),
                                    z.read(info).decode("utf-8", errors="replace"))
    else:
        raise ValueError(f"No es un ZIP ni una carpeta: {ruta}")

    env = Envio(preds={})
    if "umbral.txt" in archivos:
        try:
            u = float(archivos.pop("umbral.txt").strip().split()[0])
            if not 0.0 <= u <= 1.0:
                raise ValueError
            env.umbral, env.umbral_declarado = u, True
        except (ValueError, IndexError):
            env.avisos.append("umbral.txt inválido; se usa el umbral por defecto 0.25")
    for nombre, texto in archivos.items():
        env.preds[os.path.splitext(nombre)[0]] = _parse_lineas(texto, nombre, True, env.avisos)
    n_deg = env.avisos.count("__degenerada__")
    env.avisos = [a for a in env.avisos if a != "__degenerada__"]
    if n_deg:
        env.avisos.insert(0, f"{n_deg} cajas con ancho o alto ≤ 0 fueron ignoradas")
    if not env.preds:
        raise ValueError("El envío no contiene archivos .txt de predicciones.")
    return env


def leer_gt(carpeta: str) -> dict:
    """Etiquetas YOLO de referencia: id_imagen -> np.ndarray (M, 5): cls, xc, yc, w, h."""
    gt = {}
    for f in sorted(os.listdir(carpeta)):
        if _es_basura(f):
            continue
        with open(os.path.join(carpeta, f), "r", encoding="utf-8") as fh:
            filas = [list(map(float, l.split()[:5])) for l in fh if len(l.split()) >= 5]
        gt[os.path.splitext(f)[0]] = np.asarray(filas, dtype=np.float64).reshape(-1, 5)
    if not gt:
        raise ValueError(f"No se encontraron etiquetas en {carpeta}")
    return gt


def leer_meta(ruta: str | None, ids) -> dict:
    """id -> {'fuente':..., 'particion':...}. Sin meta: fuente = prefijo antes de '__'."""
    meta = {}
    if ruta:
        with open(ruta, newline="", encoding="utf-8") as fh:
            for r in csv.DictReader(fh):
                meta[r["id"]] = {"fuente": r.get("fuente", ""), "particion": r.get("particion", "")}
    for i in ids:
        meta.setdefault(i, {"fuente": i.split("__")[0] if "__" in i else "", "particion": ""})
    return meta


# --------------------------------------------------------------------------- núcleo
def _xyxy(b: np.ndarray) -> np.ndarray:
    xc, yc, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    return np.stack([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2], 1)


def iou_matriz(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """IoU entre cajas xc,yc,w,h normalizadas (la IoU es invariante al escalado por eje)."""
    A, B = _xyxy(a), _xyxy(b)
    ix = np.clip(np.minimum(A[:, None, 2], B[None, :, 2]) - np.maximum(A[:, None, 0], B[None, :, 0]), 0, None)
    iy = np.clip(np.minimum(A[:, None, 3], B[None, :, 3]) - np.maximum(A[:, None, 1], B[None, :, 1]), 0, None)
    inter = ix * iy
    area_a = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1])
    area_b = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-12)


def _emparejar(ious: np.ndarray) -> np.ndarray:
    """Emparejamiento voraz estilo COCO (detecciones ya ordenadas por confianza desc.).
    Devuelve (T, n_det) bool: True si la detección es TP en cada umbral de IoU."""
    T = len(UMBRALES_IOU)
    n_det, n_gt = ious.shape
    tp = np.zeros((T, n_det), dtype=bool)
    if n_det == 0 or n_gt == 0:
        return tp
    usado = np.zeros((T, n_gt), dtype=bool)
    mejor = ious.max(1)
    thr = UMBRALES_IOU[:, None]
    filas = np.arange(T)
    for d in np.nonzero(mejor >= UMBRALES_IOU[0] - 1e-12)[0]:
        cand = np.where((ious[d][None, :] >= thr - 1e-12) & ~usado, ious[d][None, :], -1.0)
        g = cand.argmax(1)
        ok = cand[filas, g] >= 0
        tp[ok, d] = True
        usado[filas[ok], g[ok]] = True
    return tp


def _ap(tp_ord: np.ndarray, n_gt: int) -> np.ndarray:
    """AP COCO (101 puntos) para cada umbral. tp_ord: (T, N) ordenado por confianza desc."""
    T = tp_ord.shape[0]
    if n_gt == 0:
        return np.full(T, np.nan)
    if tp_ord.shape[1] == 0:
        return np.zeros(T)
    tpc = np.cumsum(tp_ord, 1, dtype=np.float64)
    fpc = np.cumsum(~tp_ord, 1, dtype=np.float64)
    rec = tpc / n_gt
    pre = tpc / np.maximum(tpc + fpc, np.finfo(np.float64).eps)
    pre = np.flip(np.maximum.accumulate(np.flip(pre, 1), 1), 1)  # envolvente monótona
    out = np.zeros(T)
    for t in range(T):
        idx = np.searchsorted(rec[t], PUNTOS_RECALL, side="left")
        q = np.zeros(len(PUNTOS_RECALL))
        v = idx < len(rec[t])
        q[v] = pre[t][idx[v]]
        out[t] = q.mean()
    return out


@dataclass
class _PorClase:
    conf: list = field(default_factory=list)
    tp: list = field(default_factory=list)     # (T, n) por imagen
    img: list = field(default_factory=list)    # id de imagen por detección
    n_gt_img: dict = field(default_factory=dict)


def preparar(gt: dict, env: Envio) -> list:
    """Empareja una sola vez por imagen y clase; luego se agregan subconjuntos baratos."""
    pc = [_PorClase() for _ in range(NC)]
    for img, g in gt.items():
        p = env.preds.get(img, np.zeros((0, 6)))
        if len(p) > MAX_DET:
            p = p[np.argsort(-p[:, 5], kind="mergesort")[:MAX_DET]]
        for c in range(NC):
            gc = g[g[:, 0] == c, 1:5]
            pcd = p[p[:, 0] == c]
            pcd = pcd[np.argsort(-pcd[:, 5], kind="mergesort")]
            pc[c].n_gt_img[img] = len(gc)
            if len(pcd) == 0:
                continue
            ious = iou_matriz(pcd[:, 1:5], gc) if len(gc) else np.zeros((len(pcd), 0))
            pc[c].conf.append(pcd[:, 5])
            pc[c].tp.append(_emparejar(ious))
            pc[c].img.extend([img] * len(pcd))
    for c in range(NC):
        k = pc[c]
        k.conf = np.concatenate(k.conf) if k.conf else np.zeros(0)
        k.tp = np.concatenate(k.tp, 1) if k.tp else np.zeros((len(UMBRALES_IOU), 0), bool)
        k.img = np.asarray(k.img, dtype=object)
    return pc


def metricas(pc: list, umbral: float, subconjunto: set | None = None) -> dict:
    ap = np.full((NC, len(UMBRALES_IOU)), np.nan)
    P, R, F1, n_gt_c = np.zeros(NC), np.zeros(NC), np.zeros(NC), np.zeros(NC, int)
    for c, k in enumerate(pc):
        if subconjunto is None:
            m = np.ones(len(k.conf), bool)
            n_gt = sum(k.n_gt_img.values())
        else:
            m = np.fromiter((i in subconjunto for i in k.img), bool, len(k.img))
            n_gt = sum(v for i, v in k.n_gt_img.items() if i in subconjunto)
        conf, tp = k.conf[m], k.tp[:, m]
        orden = np.argsort(-conf, kind="mergesort")
        ap[c] = _ap(tp[:, orden], n_gt)
        sel = conf >= umbral
        n_tp = int(tp[0, sel].sum()); n_fp = int(sel.sum()) - n_tp; n_fn = n_gt - n_tp
        P[c] = n_tp / (n_tp + n_fp) if n_tp + n_fp else 0.0
        R[c] = n_tp / n_gt if n_gt else 0.0
        F1[c] = 2 * n_tp / (2 * n_tp + n_fp + n_fn) if (2 * n_tp + n_fp + n_fn) else 0.0
        n_gt_c[c] = n_gt
    validas = n_gt_c > 0
    map5095 = float(np.nanmean(ap[validas].mean(1))) if validas.any() else 0.0
    map50 = float(np.nanmean(ap[validas, 0])) if validas.any() else 0.0
    f1m = float(F1[validas].mean()) if validas.any() else 0.0
    return {
        "score": PESO_MAP * map5095 + PESO_F1 * f1m,
        "map50_95": map5095, "map50": map50, "f1_macro": f1m,
        "precision_macro": float(P[validas].mean()) if validas.any() else 0.0,
        "recall_macro": float(R[validas].mean()) if validas.any() else 0.0,
        "por_clase": {CLASES[c]: {"ap50_95": float(np.nan_to_num(ap[c].mean())), "ap50": float(np.nan_to_num(ap[c, 0])),
                                  "precision": float(P[c]), "recall": float(R[c]), "f1": float(F1[c]),
                                  "n_gt": int(n_gt_c[c])} for c in range(NC)},
    }


def barrer_umbral(pc: list, umbrales=None) -> tuple[float, float]:
    """Umbral de confianza que maximiza F1macro@0.5 (útil para elegir `umbral.txt` con el split test)."""
    umbrales = np.round(np.arange(0.05, 0.91, 0.01), 2) if umbrales is None else np.asarray(umbrales)
    f1s = []
    for k in pc:
        n_gt = sum(k.n_gt_img.values())
        if n_gt == 0:
            continue
        orden = np.argsort(-k.conf, kind="mergesort")
        conf, tp = k.conf[orden], k.tp[0, orden]
        tpc = np.concatenate([[0], np.cumsum(tp)])
        n_sel = np.searchsorted(-conf, -umbrales, side="right")   # nº de cajas con conf >= u
        n_tp = tpc[n_sel]; n_fp = n_sel - n_tp; n_fn = n_gt - n_tp
        f1s.append(2 * n_tp / np.maximum(2 * n_tp + n_fp + n_fn, 1))
    f1m = np.mean(f1s, 0)
    i = int(np.argmax(f1m))
    return float(umbrales[i]), float(f1m[i])


def evaluar(gt: dict, env: Envio, meta: dict | None = None, umbral: float | None = None) -> dict:
    u = env.umbral if umbral is None else umbral
    ids_env = set(env.preds)
    ids_gt = set(gt)
    pc = preparar(gt, env)
    res = {"umbral": u, "umbral_declarado": env.umbral_declarado or umbral is not None,
           "imagenes_gt": len(ids_gt), "imagenes_con_pred": len(ids_env & ids_gt),
           "archivos_desconocidos": len(ids_env - ids_gt),
           "total": metricas(pc, u)}
    if meta:
        parts = sorted({m["particion"] for i, m in meta.items() if i in gt and m["particion"]})
        for p in parts:
            res[p] = metricas(pc, u, {i for i in gt if meta[i]["particion"] == p})
        fuentes = sorted({m["fuente"] for i, m in meta.items() if i in gt and m["fuente"]})
        res["por_fuente"] = {f: metricas(pc, u, {i for i in gt if meta[i]["fuente"] == f}) for f in fuentes}
    avisos = list(env.avisos)
    if res["archivos_desconocidos"]:
        avisos.insert(0, f"{res['archivos_desconocidos']} archivos no corresponden a ninguna imagen evaluada (ignorados)")
    if res["imagenes_con_pred"] == 0:
        avisos.insert(0, "Ningún archivo coincide con las imágenes evaluadas: ¿nombres de archivo correctos?")
    res["avisos"] = avisos[:50] + ([f"... y {len(avisos) - 50} avisos más"] if len(avisos) > 50 else [])
    return res


# --------------------------------------------------------------------------- salida
def imprimir(res: dict, titulo: str = "") -> None:
    t = res["total"]
    print("=" * 72)
    print(f" SpaceLAB · Evaluación {titulo}".rstrip())
    print("=" * 72)
    print(f" Imágenes evaluadas: {res['imagenes_gt']}  ·  con predicciones: {res['imagenes_con_pred']}"
          f"  ·  umbral F1: {res['umbral']:.2f}{'' if res['umbral_declarado'] else ' (por defecto)'}")
    for clave in ("publico", "privado", "total"):
        if clave in res:
            m = res[clave]
            print(f" {clave.upper():8s}  SCORE {m['score']:.4f}   mAP50-95 {m['map50_95']:.4f}   "
                  f"mAP50 {m['map50']:.4f}   F1 {m['f1_macro']:.4f}")
    print("-" * 72)
    print(f" {'clase':16s} {'AP50-95':>8s} {'AP50':>8s} {'P':>7s} {'R':>7s} {'F1':>7s} {'n_gt':>7s}")
    for c, m in t["por_clase"].items():
        print(f" {c:16s} {m['ap50_95']:8.4f} {m['ap50']:8.4f} {m['precision']:7.3f} {m['recall']:7.3f} {m['f1']:7.3f} {m['n_gt']:7d}")
    if res.get("por_fuente"):
        print("-" * 72)
        print(f" {'fuente':16s} {'SCORE':>8s} {'mAP50-95':>9s} {'F1':>7s}")
        for f, m in res["por_fuente"].items():
            print(f" {f:16s} {m['score']:8.4f} {m['map50_95']:9.4f} {m['f1_macro']:7.4f}")
    if res["avisos"]:
        print("-" * 72)
        for a in res["avisos"][:15]:
            print(" AVISO:", a)
    print("=" * 72)


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Evaluador oficial SpaceLAB (Score = 0.7·mAP50-95 + 0.3·F1@0.5)")
    ap.add_argument("--gt", required=True, help="carpeta con etiquetas YOLO de referencia (.txt)")
    ap.add_argument("--meta", help="CSV con columnas id,fuente,particion (conjunto oculto)")
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--envio", help="ZIP o carpeta con las predicciones de un equipo")
    g.add_argument("--envios", help="carpeta con un ZIP (o subcarpeta) por equipo -> ranking")
    ap.add_argument("--umbral", type=float, help="forzar umbral de confianza para F1 (ignora umbral.txt)")
    ap.add_argument("--barrer-umbral", action="store_true", help="buscar el umbral que maximiza F1 (usar con test)")
    ap.add_argument("--json", help="guardar resultado completo en JSON")
    ap.add_argument("--ranking", help="CSV de salida del ranking (con --envios)")
    a = ap.parse_args(argv)

    gt = leer_gt(a.gt)
    meta = leer_meta(a.meta, gt) if a.meta else leer_meta(None, gt)

    if a.envio:
        env = leer_envio(a.envio)
        res = evaluar(gt, env, meta, a.umbral)
        if a.barrer_umbral:
            u, f = barrer_umbral(preparar(gt, env))
            res["umbral_optimo"] = {"umbral": u, "f1_macro": f}
        imprimir(res, os.path.basename(a.envio.rstrip("/")))
        if a.barrer_umbral:
            print(f" Umbral que maximiza F1macro@0.5: {res['umbral_optimo']['umbral']:.2f} "
                  f"(F1 = {res['umbral_optimo']['f1_macro']:.4f}) -> escríbalo en umbral.txt")
        if a.json:
            with open(a.json, "w", encoding="utf-8") as fh:
                json.dump(res, fh, ensure_ascii=False, indent=2)
        return 0

    filas = []
    for nombre in sorted(os.listdir(a.envios)):
        r = os.path.join(a.envios, nombre)
        if nombre.startswith(".") or not (os.path.isdir(r) or nombre.lower().endswith(".zip")):
            continue
        equipo = os.path.splitext(nombre)[0]
        try:
            res = evaluar(gt, leer_envio(r), meta, a.umbral)
        except Exception as e:  # noqa: BLE001
            print(f"[{equipo}] ERROR: {e}", file=sys.stderr)
            continue
        fila = {"equipo": equipo}
        for clave in ("total", "publico", "privado"):
            if clave in res:
                for k in ("score", "map50_95", "map50", "f1_macro"):
                    fila[f"{clave}_{k}"] = round(res[clave][k], 5)
        filas.append(fila)
        print(f"[{equipo}] total score {res['total']['score']:.4f}")
    clave = "privado_score" if filas and "privado_score" in filas[0] else "total_score"
    desempate = clave.replace("score", "map50")
    filas.sort(key=lambda f: (-f[clave], -f[desempate]))
    print("\n RANKING (" + clave + ")")
    for i, f in enumerate(filas, 1):
        print(f" {i:2d}. {f['equipo']:30s} {f[clave]:.4f}")
    if a.ranking and filas:
        with open(a.ranking, "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=list(filas[0].keys()))
            w.writeheader(); w.writerows(filas)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile formato_envio.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
formato_envio.py — Ayuda para el formato de envío · SpaceLAB Hackatón FAC+PUJ · GeoVision Colombia
===================================================================================================

Sirve con CUALQUIER detector (YOLO de cualquier versión, Faster R-CNN, RT-DETR, Detectron2, MMDetection...).
Solo se encarga del formato: no toca su modelo ni usa las respuestas del conjunto oculto.

1) Desde su código (las cajas en PÍXELES, formato x1 y1 x2 y2, como las entregan casi todos los frameworks):

    from formato_envio import escribir_txt, crear_zip
    # detecciones de UNA imagen: lista de (clase, x1, y1, x2, y2, confianza), clase entera 0..4
    escribir_txt("preds/sl_3fa9c2d1.txt", detecciones, ancho_imagen, alto_imagen)
    ...
    crear_zip("preds/", "mi_envio.zip", umbral=0.30)

2) Desde la terminal:

    # arma el ZIP con los .txt de una carpeta (+ umbral.txt)
    python formato_envio.py zip --carpeta preds/ --salida mi_envio.zip --umbral 0.30

    # revisa un envío ANTES de subirlo: nombres, cantidad de archivos y formato de cada línea
    python formato_envio.py verificar --envio mi_envio.zip --imagenes spacelab_dataset/images/oculto

Formato oficial (lo que escribe este archivo): un .txt por imagen, mismo nombre que la imagen, una línea por
detección `clase x_centro y_centro ancho alto confianza`, coordenadas normalizadas 0-1. Máximo 300 por imagen.
La verificación usa el mismo lector que el evaluador oficial (evaluar.py, que debe estar en la misma carpeta).
"""
from __future__ import annotations

import argparse
import os
import sys
import zipfile

CLASES = ["small_boat", "cargo_ship", "fishing_vessel", "warship", "work_vessel"]
MAX_DET = 300
EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


# --------------------------------------------------------------------------- escribir
def linea(clase: int, x1: float, y1: float, x2: float, y2: float, conf: float, ancho: float, alto: float) -> str | None:
    """Una detección en píxeles (x1,y1,x2,y2) -> línea oficial. Devuelve None si la caja queda vacía."""
    x1, x2 = sorted((min(max(float(x1), 0.0), ancho), min(max(float(x2), 0.0), ancho)))
    y1, y2 = sorted((min(max(float(y1), 0.0), alto), min(max(float(y2), 0.0), alto)))
    w, h = (x2 - x1) / ancho, (y2 - y1) / alto
    if w <= 0 or h <= 0:
        return None
    c = int(clase)
    if not 0 <= c < len(CLASES):
        raise ValueError(f"clase {c} fuera de rango 0..{len(CLASES) - 1}")
    conf = min(max(float(conf), 0.0), 1.0)
    return f"{c} {(x1 + x2) / 2 / ancho:.6f} {(y1 + y2) / 2 / alto:.6f} {w:.6f} {h:.6f} {conf:.5f}"


def escribir_txt(ruta_txt: str, detecciones, ancho: float, alto: float) -> int:
    """Escribe el .txt de una imagen. detecciones: iterable de (clase, x1, y1, x2, y2, confianza) en píxeles.
    Se guardan como máximo las 300 de mayor confianza. Crea el archivo aunque no haya detecciones."""
    dets = sorted(detecciones, key=lambda d: -float(d[5]))[:MAX_DET]
    lineas = [l for l in (linea(*d, ancho, alto) for d in dets) if l]
    os.makedirs(os.path.dirname(os.path.abspath(ruta_txt)), exist_ok=True)
    with open(ruta_txt, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lineas))
    return len(lineas)


def crear_zip(carpeta: str, ruta_zip: str, umbral: float | None = None) -> int:
    """Empaqueta todos los .txt de `carpeta` (sin subcarpetas) y, si se indica, umbral.txt."""
    txts = sorted(f for f in os.listdir(carpeta) if f.endswith(".txt") and f != "umbral.txt" and not f.startswith("."))
    if not txts:
        raise ValueError(f"No hay archivos .txt en {carpeta}")
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for f in txts:
            z.write(os.path.join(carpeta, f), f)
        if umbral is not None:
            if not 0 <= umbral <= 1:
                raise ValueError("el umbral debe estar entre 0 y 1")
            z.writestr("umbral.txt", f"{umbral:.2f}\n")
    return len(txts)


# --------------------------------------------------------------------------- verificar
def verificar(envio: str, carpeta_imagenes: str | None = None) -> bool:
    """Revisa un envío (ZIP o carpeta) con el mismo lector del evaluador oficial. Devuelve True si se puede subir."""
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    try:
        import evaluar as EV
    except ImportError:
        sys.exit("Para verificar, ponga evaluar.py en la misma carpeta que formato_envio.py")
    ok = True
    try:
        env = EV.leer_envio(envio)
    except ValueError as e:
        print(f"✗ {e}")
        return False
    n_det = sum(len(v) for v in env.preds.values())
    print(f"Archivos de predicción: {len(env.preds)} · detecciones válidas: {n_det}")
    print(f"Umbral para F1: {env.umbral:.2f} ({'umbral.txt' if env.umbral_declarado else 'no hay umbral.txt: se usará 0.25'})")
    muchos = [k for k, v in env.preds.items() if len(v) > MAX_DET]
    if muchos:
        print(f"! {len(muchos)} archivos tienen más de {MAX_DET} detecciones: solo cuentan las {MAX_DET} de mayor confianza")
    if carpeta_imagenes:
        imgs = {os.path.splitext(f)[0] for f in os.listdir(carpeta_imagenes)
                if os.path.splitext(f)[1].lower() in EXT_IMG and not f.startswith(".")}
        sobran = sorted(set(env.preds) - imgs)
        faltan = sorted(imgs - set(env.preds))
        print(f"Imágenes en {carpeta_imagenes}: {len(imgs)} · con archivo: {len(imgs & set(env.preds))}")
        if sobran:
            ok = False
            print(f"✗ {len(sobran)} archivos no corresponden a ninguna imagen (revise los nombres), p. ej.: {', '.join(sobran[:5])}")
        if faltan:
            print(f"! {len(faltan)} imágenes sin archivo: se califican como 'sin detecciones', p. ej.: {', '.join(faltan[:5])}")
        if not imgs & set(env.preds):
            ok = False
            print("✗ Ningún archivo coincide con las imágenes: el envío sería rechazado.")
    for a in env.avisos[:15]:
        print(f"! {a}")
    if len(env.avisos) > 15:
        print(f"! ... y {len(env.avisos) - 15} avisos más")
    print("✔ El envío se puede subir." if ok else "✗ Corrija lo anterior antes de subir.")
    return ok


# --------------------------------------------------------------------------- terminal
def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    z = sub.add_parser("zip", help="empaquetar una carpeta de .txt")
    z.add_argument("--carpeta", required=True); z.add_argument("--salida", required=True)
    z.add_argument("--umbral", type=float, default=None)
    v = sub.add_parser("verificar", help="revisar un envío antes de subirlo")
    v.add_argument("--envio", required=True, help="ZIP o carpeta")
    v.add_argument("--imagenes", default=None, help="carpeta images/oculto (para revisar los nombres)")
    a = ap.parse_args()
    if a.cmd == "zip":
        n = crear_zip(a.carpeta, a.salida, a.umbral)
        print(f"✔ {a.salida}: {n} archivos" + (f" + umbral.txt ({a.umbral:.2f})" if a.umbral is not None else ""))
        return 0
    return 0 if verificar(a.envio, a.imagenes) else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile entrenar_baseline_frcnn.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
entrenar_baseline_frcnn.py — Segundo baseline: Faster R-CNN (torchvision) · SpaceLAB Hackatón FAC+PUJ
=====================================================================================================

Un detector clásico de dos etapas, en PyTorch puro (sin Ultralytics), con pesos preentrenados en COCO.
Hace el mismo ciclo que el baseline de YOLO:
  1. Entrena sobre images/train leyendo directamente las etiquetas en formato YOLO.
  2. Predice images/test y calcula el PUNTAJE OFICIAL local con evaluar.py; elige el umbral que maximiza F1.
  3. Predice images/oculto y genera el ZIP de envío (con formato_envio.py).

Los baselines son solo puntos de partida para verificar que su flujo funciona de punta a punta.
No son una recomendación: pueden usar cualquier modelo o framework.

Necesita: torch y torchvision (ya vienen en Colab y Kaggle), pillow y numpy. evaluar.py y formato_envio.py
deben estar en la misma carpeta.

Ejemplos:
  # Colab/Kaggle T4, ~1.5 h de entrenamiento como máximo
  python entrenar_baseline_frcnn.py --datos /content/spacelab_dataset --horas 1.5

  # Prueba rápida (5 % de los datos, 1 época, 640 px)
  python entrenar_baseline_frcnn.py --datos ./spacelab_dataset --fraccion 0.05 --epocas 1 --imgsz 640

  # Solo generar el envío con unos pesos ya entrenados
  python entrenar_baseline_frcnn.py --datos ./spacelab_dataset --pesos runs_spacelab/frcnn/weights/last.pt

Memoria: algunas imágenes tienen cientos de barcos y disparan el consumo de memoria al entrenar.
Si aparece `CUDA out of memory`, use un --batch pequeño.
"""
from __future__ import annotations

import argparse
import contextlib
import math
import random
import sys
import time
from pathlib import Path

AQUI = Path(__file__).resolve().parent
sys.path.insert(0, str(AQUI))
import evaluar as EV  # noqa: E402
import formato_envio as FE  # noqa: E402

import torch  # noqa: E402
from PIL import Image  # noqa: E402

EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
NC = len(EV.CLASES)


# --------------------------------------------------------------------------- datos
class DatasetYOLO(torch.utils.data.Dataset):
    """Imágenes + etiquetas YOLO (clase xc yc w h normalizados) -> formato de torchvision (cajas x1y1x2y2 en píxeles)."""

    def __init__(self, carpeta_img: Path, carpeta_lab: Path | None = None, entrenar: bool = False,
                 fraccion: float = 1.0, semilla: int = 0):
        self.imgs = sorted(p for p in Path(carpeta_img).iterdir()
                           if p.suffix.lower() in EXT and not p.name.startswith("."))
        if fraccion < 1.0:
            random.Random(semilla).shuffle(self.imgs)
            self.imgs = sorted(self.imgs[:max(1, int(len(self.imgs) * fraccion))])
        self.lab, self.entrenar = carpeta_lab, entrenar

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        from torchvision.transforms.functional import to_tensor
        p = self.imgs[i]
        im = Image.open(p).convert("RGB")
        W, H = im.size
        x = to_tensor(im)
        cajas, clases = [], []
        if self.lab is not None:
            f = Path(self.lab) / (p.stem + ".txt")
            if f.exists():
                for l in f.read_text().splitlines():
                    v = l.split()
                    if len(v) >= 5:
                        c, xc, yc, w, h = int(float(v[0])), *map(float, v[1:5])
                        cajas.append([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H])
                        clases.append(c + 1)  # 0 es el fondo en torchvision
        b = torch.tensor(cajas, dtype=torch.float32).reshape(-1, 4)
        b[:, 0::2] = b[:, 0::2].clamp(0, W)
        b[:, 1::2] = b[:, 1::2].clamp(0, H)
        if self.entrenar:  # imágenes cenitales: no hay "arriba" ni "abajo"
            if random.random() < 0.5:
                x = x.flip(-1); b[:, [0, 2]] = W - b[:, [2, 0]]
            if random.random() < 0.5:
                x = x.flip(-2); b[:, [1, 3]] = H - b[:, [3, 1]]
        ok = ((b[:, 2] - b[:, 0]) >= 1) & ((b[:, 3] - b[:, 1]) >= 1)
        objetivo = {"boxes": b[ok], "labels": torch.tensor(clases, dtype=torch.int64)[ok]}
        return x, objetivo, p, (W, H)


def juntar(lote):
    return tuple(zip(*lote))


# --------------------------------------------------------------------------- modelo
def construir_modelo(arquitectura: str, imgsz: int, preentrenado: bool = True):
    from torchvision.models import detection as D
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    opciones = {
        "resnet50_v2": (D.fasterrcnn_resnet50_fpn_v2, "FasterRCNN_ResNet50_FPN_V2_Weights"),
        "resnet50": (D.fasterrcnn_resnet50_fpn, "FasterRCNN_ResNet50_FPN_Weights"),
        "mobilenet": (D.fasterrcnn_mobilenet_v3_large_fpn, "FasterRCNN_MobileNet_V3_Large_FPN_Weights"),
    }
    fn, pesos = opciones[arquitectura]
    pesos = getattr(D, pesos).DEFAULT if preentrenado else None
    # min_size = max_size = imgsz -> el lado largo de cada imagen queda en imgsz (como imgsz en YOLO)
    m = fn(weights=pesos, min_size=imgsz, max_size=imgsz, box_detections_per_img=300)
    entrada = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(entrada, NC + 1)
    return m


def elegir_device(pedido: str) -> torch.device:
    if pedido != "auto":
        return torch.device(pedido if not pedido.isdigit() else f"cuda:{pedido}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        print("Usando Apple Silicon (mps)")
        return torch.device("mps")
    print("⚠️  Sin GPU: el entrenamiento en CPU es MUY lento. Use Colab/Kaggle o --fraccion 0.05 --imgsz 640.")
    return torch.device("cpu")


def usa_amp(dev: torch.device) -> bool:
    # precisión mixta solo en GPUs NVIDIA con núcleos tensor (T4, A100, RTX...). En GPUs viejas es más lenta.
    return dev.type == "cuda" and torch.cuda.get_device_capability(dev)[0] >= 7


# --------------------------------------------------------------------------- entrenamiento
def entrenar(m, dl, dev, epocas: int, horas: float | None, lr: float, carpeta_pesos: Path, meta: dict) -> Path:
    m.to(dev).train()
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    amp = usa_amp(dev)
    escalador = torch.cuda.amp.GradScaler(enabled=amp)
    limite = horas * 3600 if horas else None
    total = epocas * len(dl)
    calentamiento = min(300, total // 5 + 1)
    carpeta_pesos.mkdir(parents=True, exist_ok=True)
    ultimo = carpeta_pesos / "last.pt"
    t0, it, fin = time.time(), 0, False
    print(f"Entrenando: {len(dl.dataset)} imágenes · lote {dl.batch_size} · "
          f"{'límite ' + str(horas) + ' h' if limite else str(epocas) + ' épocas'} · precisión mixta: {amp}")
    for ep in range(1, epocas + 1):
        suma, n, t_ep = 0.0, 0, time.time()
        for imgs, objetivos, _, _ in dl:
            progreso = min(1.0, (time.time() - t0) / limite) if limite else it / total
            factor = min(1.0, (it + 1) / calentamiento) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * progreso)))
            for g in opt.param_groups:
                g["lr"] = lr * factor
            imgs = [x.to(dev) for x in imgs]
            objetivos = [{k: v.to(dev) for k, v in o.items()} for o in objetivos]
            ctx = torch.autocast("cuda", dtype=torch.float16) if amp else contextlib.nullcontext()
            with ctx:
                perdidas = m(imgs, objetivos)
                perdida = sum(perdidas.values())
            if not torch.isfinite(perdida):
                print("  pérdida no finita: se salta el lote"); opt.zero_grad(); it += 1; continue
            opt.zero_grad()
            escalador.scale(perdida).backward()
            escalador.step(opt)
            escalador.update()
            suma += float(perdida); n += 1; it += 1
            if it % 50 == 0:
                print(f"  época {ep} · iter {it} · pérdida {suma / n:.3f} · lr {lr * factor:.5f} · "
                      f"{(time.time() - t0) / 60:.1f} min", flush=True)
            if limite and time.time() - t0 > limite:
                fin = True
                break
        torch.save({"modelo": m.state_dict(), **meta}, ultimo)
        print(f"Época {ep} terminada en {(time.time() - t_ep) / 60:.1f} min · pérdida media {suma / max(n, 1):.3f} "
              f"· pesos: {ultimo}", flush=True)
        if fin:
            print("Se alcanzó el límite de tiempo.")
            break
    return ultimo


# --------------------------------------------------------------------------- predicción
@torch.no_grad()
def predecir_carpeta(m, carpeta: Path, salida: Path, dev, lote: int = 4, workers: int = 2) -> int:
    """Escribe un .txt por imagen (aunque no tenga detecciones) en formato oficial."""
    m.to(dev).eval()
    m.roi_heads.score_thresh = 0.001   # enviar también las cajas de baja confianza (el mAP las aprovecha)
    m.roi_heads.detections_per_img = 300
    salida.mkdir(parents=True, exist_ok=True)
    ds = DatasetYOLO(carpeta)
    dl = torch.utils.data.DataLoader(ds, batch_size=lote, shuffle=False, num_workers=workers, collate_fn=juntar)
    amp = usa_amp(dev)
    n = 0
    for imgs, _, rutas, tams in dl:
        ctx = torch.autocast("cuda", dtype=torch.float16) if amp else contextlib.nullcontext()
        with ctx:
            salidas = m([x.to(dev) for x in imgs])
        for o, ruta, (W, H) in zip(salidas, rutas, tams):
            dets = [(int(c) - 1, *b, s) for b, c, s in
                    zip(o["boxes"].float().tolist(), o["labels"].tolist(), o["scores"].float().tolist())]
            FE.escribir_txt(str(salida / (ruta.stem + ".txt")), dets, W, H)
            n += 1
        if n % 200 < lote:
            print(f"  {n}/{len(ds)} imágenes", flush=True)
    return n


# --------------------------------------------------------------------------- main
def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--datos", required=True, help="carpeta spacelab_dataset")
    ap.add_argument("--arquitectura", default="resnet50_v2", choices=["resnet50_v2", "resnet50", "mobilenet"])
    ap.add_argument("--imgsz", type=int, default=1024, help="lado largo de la imagen al entrenar/predecir")
    ap.add_argument("--epocas", type=int, default=12)
    ap.add_argument("--horas", type=float, default=None, help="límite de tiempo de entrenamiento")
    ap.add_argument("--batch", type=int, default=None, help="imágenes por lote (defecto: 4 en GPU, 1 en mps/cpu)")
    ap.add_argument("--lr", type=float, default=None, help="tasa de aprendizaje (defecto: 0.0025 × lote)")
    ap.add_argument("--fraccion", type=float, default=1.0, help="fracción de train a usar (pruebas rápidas)")
    ap.add_argument("--device", default="auto", help="auto | 0 | cuda | mps | cpu")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--proyecto", default="runs_spacelab")
    ap.add_argument("--nombre", default="frcnn")
    ap.add_argument("--pesos", default=None, help="saltar el entrenamiento y usar estos pesos (last.pt)")
    a = ap.parse_args()

    datos = Path(a.datos).resolve()
    if not (datos / "images" / "train").exists():
        sys.exit(f"No encuentro {datos / 'images' / 'train'}")
    dev = elegir_device(a.device)
    torch.manual_seed(0); random.seed(0)
    salida = Path(a.proyecto).resolve() / a.nombre
    t0 = time.time()

    # ------------------------------------------------------------------ 1. entrenamiento
    if a.pesos:
        ck = torch.load(a.pesos, map_location="cpu")
        m = construir_modelo(ck.get("arquitectura", a.arquitectura), ck.get("imgsz", a.imgsz), preentrenado=False)
        m.load_state_dict(ck["modelo"])
    else:
        lote = a.batch or (4 if dev.type == "cuda" else 1)
        m = construir_modelo(a.arquitectura, a.imgsz)
        ds = DatasetYOLO(datos / "images" / "train", datos / "labels" / "train", entrenar=True, fraccion=a.fraccion)
        dl = torch.utils.data.DataLoader(ds, batch_size=lote, shuffle=True, num_workers=a.workers,
                                         collate_fn=juntar, drop_last=True, persistent_workers=a.workers > 0)
        entrenar(m, dl, dev, a.epocas, a.horas, a.lr or 0.0025 * lote, salida / "weights",
                 {"arquitectura": a.arquitectura, "imgsz": a.imgsz})
        print(f"\nEntrenamiento terminado en {(time.time() - t0) / 60:.1f} min")

    # ------------------------------------------------------------------ 2. puntaje local en test
    print("\nPrediciendo images/test para calcular el puntaje oficial local ...")
    pred_test = salida / "pred_test"
    predecir_carpeta(m, datos / "images" / "test", pred_test, dev, workers=a.workers)
    gt_test = EV.leer_gt(str(datos / "labels" / "test"))
    env_test = EV.leer_envio(str(pred_test))
    u, _ = EV.barrer_umbral(EV.preparar(gt_test, env_test))
    res = EV.evaluar(gt_test, env_test, EV.leer_meta(None, gt_test), umbral=u)
    EV.imprimir(res, "split test (local)")
    print(f" Umbral elegido (maximiza F1 en test): {u:.2f}")

    # ------------------------------------------------------------------ 3. envío sobre el conjunto oculto
    oculto = datos / "images" / "oculto"
    if not oculto.exists():
        print("No hay images/oculto; no se genera envío.")
        return 0
    print("\nPrediciendo images/oculto ...")
    pred_oc = salida / "envio"
    n = predecir_carpeta(m, oculto, pred_oc, dev, workers=a.workers)
    zpath = salida / f"envio_{a.nombre}.zip"
    FE.crear_zip(str(pred_oc), str(zpath), umbral=round(u, 2))
    print(f"\n✅ Envío listo: {zpath}  ({n} imágenes, umbral {u:.2f})")
    print(f"   Puntaje local esperado (test): {res['total']['score']:.4f}  ·  tiempo total {(time.time() - t0) / 60:.1f} min")
    print("   Revíselo con:  python formato_envio.py verificar --envio", zpath, "--imagenes", oculto)
    return 0


if __name__ == "__main__":
    sys.exit(main())


### 3. Entrenar, evaluar en test y generar el envío
`--horas` limita el tiempo de entrenamiento (en las pruebas de los organizadores, 1.5 h ≈ 4 épocas a 1024 px en una GTX 1080 Ti; en una T4 es parecido).
Pruebe primero con `--fraccion 0.05 --epocas 1` para verificar que todo corre.
Si el entrenamiento se cae por `CUDA out of memory`, agregue `--batch` con un valor pequeño.
Los pesos se guardan al final de cada época en `runs_spacelab/frcnn/weights/last.pt` (cópielos a Drive: una sesión gratuita puede desconectarse). Con `--pesos` se genera el envío sin volver a entrenar.

In [ ]:
HORAS = 1.5
!python entrenar_baseline_frcnn.py --datos "{DATOS}" --horas {HORAS} --nombre frcnn

### 4. Revisar el envío antes de subirlo

In [ ]:
!python formato_envio.py verificar --envio runs_spacelab/frcnn/envio_frcnn.zip --imagenes "{DATOS}/images/oculto"

### 5. Descargar el ZIP y subirlo al leaderboard (sección 📤 Enviar)

In [ ]:
ZIP = 'runs_spacelab/frcnn/envio_frcnn.zip'
print(ZIP, os.path.getsize(ZIP)//1024, 'KB')
try:
    from google.colab import files; files.download(ZIP)      # Colab
except ImportError:
    print('En Kaggle: descárguelo desde el panel Output (/kaggle/working).')

### Ideas para mejorar
Las del documento del reto (resolución, tiles, augmentations, desbalance de clases, más tiempo de entrenamiento...).
* Ojo: el mAP premia enviar muchas cajas de baja confianza, pero el F1 solo cuenta las que superan su `umbral.txt`.